# NeuroGraph · Phase 6 + 8 · university intake, search index, policy graph, literature graph, agent
Settings: Accelerator **GPU T4 x2**, Internet **On**. No inputs.
Secrets (**Add-ons → Secrets**, attached): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`.
Fetches only the approved public pages (one request every 2 s); raw pages stay in `/tmp`.

## 1. Protect preinstalled packages

In [ ]:
from importlib.metadata import version

def write_constraints(pkgs: list[str], path: str) -> None:
    with open(path, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in pkgs))

write_constraints(["torch", "numpy", "scipy", "scikit-learn", "mne"], "/tmp/keep.txt")
print(open("/tmp/keep.txt").read())

## 2. Get the latest code + install

In [ ]:
import os
R = "/kaggle/working/neurograph"
if not os.path.isdir(R):
    !git clone -q https://github.com/stuBBornButterfly/neurograph.git {R}
!git -C {R} pull -q
!git -C {R} log --oneline -1
%cd {R}
!pip install -q -r requirements.txt -c /tmp/keep.txt
!pip install -q -e . -c /tmp/keep.txt

## 3. spaCy model + all tests (PII, safety, read-only Cypher)

In [ ]:
!python -m spacy download en_core_web_sm -q 2>&1 | tail -1
!cd /kaggle/working/neurograph && python -m pytest tests -q -p no:cacheprovider 2>&1 | tail -3

## 4. AuraDB credentials from Kaggle Secrets (never printed)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for key in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
    os.environ[key] = secrets.get_secret(key)
print("loaded:", [k for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"] if os.environ.get(k)])

## 5. Step 6.1: fetch -> clean -> PII redaction -> chunks

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.ingest_pages 2>&1 | grep -v -i warning

## 6. Step 6.2: search index (dense BGE in Qdrant + BM25) and retrieval test

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.build_index 2>&1 | grep -v -i 'warn\|it/s'

## 7. Step 6.3: The university policy graph in Neo4j (documents, sections, services, rules)

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.build_policy_graph

## 7b. Step 6.4-6.6: PubMed claims -> NLI check -> literature graph -> replication vs our data

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.build_literature_graph 2>&1 | grep -v -i 'warn\|it/s\|Loading'

## 8. Step 8: agent evaluation (safety -> router -> retrieval/graph -> Qwen2.5-3B -> verifier)

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.run_agent_eval 2>&1 | grep -v -i 'warn\|it/s\|Loading'